In [5]:
import torch
import torch.nn as nn
import numpy as np
from rollout_loader import load_rollouts
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
import glob
import pickle
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
from ray import tune
from ray.tune.schedulers import ASHAScheduler
from pathlib import Path


In [ ]:
TRAINING_DIR = Path("training_trajectories")
TESTING_DIR = Path("testing_trajectories")

training_trajectory_files = sorted(TRAINING_DIR.glob("data_*.pkl"))
print(f"\nFound {len(training_trajectory_files)} training trajectory files")

testing_trajectory_files = sorted(TESTING_DIR.glob("data_*.pkl"))
print(f"Found {len(testing_trajectory_files)} testing trajectory files")


Found 140 training trajectory files
Found 60 testing trajectory files


In [ ]:
training_trajectories = []
for f in training_trajectory_files:
    with open(f, "rb") as file:
        data = pickle.load(file)
    
    q_mes = np.array(data['q_mes_all'])
    q_des = np.array(data['q_d_all'])
    qd_des = np.array(data['qd_d_all'])
    final_target = np.array(data['desired_cartesian_pos']) if 'desired_cartesian_pos' in data else np.array(data['cart_pos_all'][-1])
    
    final_targets = np.tile(final_target, (q_mes.shape[0], 1))
    
    X = np.hstack([q_mes, final_targets]) 
    y = np.hstack([q_des, qd_des]) 
    
    training_trajectories.append({
        'X': X,
        'y': y,
        'filename': f,
        'target': final_target
    })
    print(f"  {f}: {X.shape[0]} timesteps, target: {final_target}")

print(len(training_trajectories))

  training_trajectories/data_0.pkl: 2697 timesteps, target: [ 0.86818996 -0.36108647  0.58559515]
  training_trajectories/data_1.pkl: 2916 timesteps, target: [ 0.27452142 -0.51458648  0.69294092]
  training_trajectories/data_10.pkl: 2214 timesteps, target: [ 0.61042686 -0.37319925  0.21707923]
  training_trajectories/data_100.pkl: 1677 timesteps, target: [ 0.08035463 -0.62240846  0.22287227]
  training_trajectories/data_101.pkl: 1579 timesteps, target: [ 0.63766204 -0.07897095  0.29141589]
  training_trajectories/data_102.pkl: 3342 timesteps, target: [0.54549731 0.23179789 0.6988629 ]
  training_trajectories/data_103.pkl: 1919 timesteps, target: [ 0.67441251 -0.56915694  0.59860775]
  training_trajectories/data_104.pkl: 1573 timesteps, target: [ 0.80634781 -0.03923755  0.29513091]
  training_trajectories/data_105.pkl: 2564 timesteps, target: [0.71946631 0.57030179 0.40934195]
  training_trajectories/data_106.pkl: 2255 timesteps, target: [ 0.76252111 -0.12713407  0.1939764 ]
  training_

In [ ]:
testing_trajectories = []
for f in testing_trajectory_files:
    with open(f, "rb") as file:
        data = pickle.load(file)
    
    q_mes = np.array(data['q_mes_all'])
    q_des = np.array(data['q_d_all'])
    qd_des = np.array(data['qd_d_all'])
    final_target = np.array(data['desired_cartesian_pos']) if 'desired_cartesian_pos' in data else np.array(data['cart_pos_all'][-1])
    
    final_targets = np.tile(final_target, (q_mes.shape[0], 1))
    
    X = np.hstack([q_mes, final_targets])  
    y = np.hstack([q_des, qd_des])     
    
    testing_trajectories.append({
        'X': X,
        'y': y,
        'filename': f,
        'target': final_target
    })
    print(f"  {f}: {X.shape[0]} timesteps, target: {final_target}")


  testing_trajectories/data_140.pkl: 1797 timesteps, target: [0.51713963 0.22816739 0.5308299 ]
  testing_trajectories/data_141.pkl: 2691 timesteps, target: [ 0.25400393 -0.92449067  0.40175601]
  testing_trajectories/data_142.pkl: 1881 timesteps, target: [0.45540661 0.45529351 0.71844343]
  testing_trajectories/data_143.pkl: 2549 timesteps, target: [ 0.78291222 -0.0688808   0.05256691]
  testing_trajectories/data_144.pkl: 1864 timesteps, target: [ 0.74607001 -0.24043706  0.1241922 ]
  testing_trajectories/data_145.pkl: 2317 timesteps, target: [0.11950396 0.69918449 0.21659574]
  testing_trajectories/data_146.pkl: 1962 timesteps, target: [ 0.39875653 -0.74657042  0.52713915]
  testing_trajectories/data_147.pkl: 1857 timesteps, target: [ 0.03723145 -0.60049033  0.63993287]
  testing_trajectories/data_148.pkl: 3211 timesteps, target: [ 0.40807257 -0.43955511  0.19453775]
  testing_trajectories/data_149.pkl: 1526 timesteps, target: [ 0.78445371 -0.26258057  0.16398744]
  testing_trajector

In [ ]:
X_train_list = [training_trajectories[i]['X'] for i in range(len(training_trajectories))]
y_train_list = [training_trajectories[i]['y'] for i in range(len(training_trajectories))]

X_test_list = [testing_trajectories[i]['X'] for i in range(len(testing_trajectories))]
y_test_list = [testing_trajectories[i]['y'] for i in range(len(testing_trajectories))]

X_train = np.vstack(X_train_list)
y_train = np.vstack(y_train_list)
X_test = np.vstack(X_test_list)
y_test = np.vstack(y_test_list)

print(f"\nTraining data: {X_train.shape[0]} timesteps from {len(training_trajectories)} trajectories")
print(f"Test data: {X_test.shape[0]} timesteps from {len(testing_trajectories)} trajectories")



Training data: 348524 timesteps from 140 trajectories
Test data: 145995 timesteps from 60 trajectories


In [ ]:
# Normalize - ONLY using training data statistics

X_mean = X_train.mean(axis=0)
X_std = X_train.std(axis=0)
y_mean = y_train.mean(axis=0)
y_std = y_train.std(axis=0)

X_std_safe = np.where(X_std == 0, 1, X_std)
y_std_safe = np.where(y_std == 0, 1, y_std)

# Apply normalization
X_train_norm = (X_train - X_mean) / X_std_safe
y_train_norm = (y_train - y_mean) / y_std_safe

X_test_norm = (X_test - X_mean) / X_std_safe
y_test_norm = (y_test - y_mean) / y_std_safe

print("Normalization complete using training statistics only")

Normalization complete using training statistics only


In [11]:

X_train, X_temp, y_train, y_temp = train_test_split(X_train_norm, y_train_norm, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

In [12]:
# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

In [9]:
class DeepMLP_Tunable(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_sizes):
        super().__init__()
        layers = []
        last_dim = input_dim
        for h in hidden_sizes:
            layers.append(nn.Linear(last_dim, h))
            layers.append(nn.ReLU())
            last_dim = h
        layers.append(nn.Linear(last_dim, output_dim))
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x)

In [10]:
def train_mlp_tune(config, X_train, y_train, X_val, y_val, input_dim=10, output_dim=14):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = DeepMLP_Tunable(input_dim, output_dim, config["hidden_sizes"]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])
    criterion = nn.MSELoss()

    X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_val_tensor = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_val_tensor = torch.tensor(y_val, dtype=torch.float32).to(device)

    train_ds = TensorDataset(X_train_tensor, y_train_tensor)
    train_dl = DataLoader(train_ds, batch_size=int(config["batch_size"]), shuffle=True)

    for epoch in range(int(config["epochs"])):
        model.train()
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)  
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_preds = model(X_val_tensor)
            val_loss = criterion(val_preds, y_val_tensor).item()

        tune.report(loss=val_loss)


In [11]:
# Training function for Ray Tune
def train_mlp_tune(config, X_train_tensor, y_train_tensor, X_val_tensor, y_val_tensor):
    """Training function that Ray Tune will call"""
    from ray.air import session
    import torch
    import torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    X_train_dev = X_train_tensor.to(device)
    y_train_dev = y_train_tensor.to(device)
    X_val_dev = X_val_tensor.to(device)
    y_val_dev = y_val_tensor.to(device)
    
    # Create model
    input_dim = X_train_tensor.shape[1]
    output_dim = y_train_tensor.shape[1]
    model = DeepMLP_Tunable(input_dim, output_dim, config["hidden_sizes"]).to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])
    criterion = nn.MSELoss()
    
    # Create dataloader
    train_ds = TensorDataset(X_train_dev, y_train_dev)
    train_dl = DataLoader(train_ds, batch_size=int(config["batch_size"]), shuffle=True)
    
    # Training loop
    for epoch in range(int(config["epochs"])):
        model.train()
        train_loss = 0
        for xb, yb in train_dl:
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * xb.size(0)
        
        train_loss /= len(train_ds)
        
        # Validation
        model.eval()
        with torch.no_grad():
            val_preds = model(X_val_dev)
            val_loss = criterion(val_preds, y_val_dev).item()
        
        session.report({"loss": val_loss, "train_loss": train_loss})


In [12]:
# Define search space
search_space = {
    "hidden_sizes": tune.choice([[64, 64], [128, 128], [64, 128, 64], [128, 256, 128]]),
    "lr": tune.loguniform(1e-4, 1e-2),
    "batch_size": tune.choice([64, 128, 256]),
    "epochs": tune.choice([50, 100, 150]),
}


In [13]:

# Run Ray Tune
print("Starting Ray Tune hyperparameter search...")

scheduler = ASHAScheduler(
    max_t=150,
    grace_period=10,
    reduction_factor=2
)

tuner = tune.Tuner(
    tune.with_parameters(
        train_mlp_tune,
        X_train_tensor=X_train_tensor,
        y_train_tensor=y_train_tensor,
        X_val_tensor=X_val_tensor,
        y_val_tensor=y_val_tensor,
    ),
    param_space=search_space,
    tune_config=tune.TuneConfig(
        metric="loss",
        mode="min",
        scheduler=scheduler,
        num_samples=10,
    ),
)

results = tuner.fit()


2025-10-30 11:50:29,100	INFO tune.py:1041 -- Total run time: 2719.72 seconds (2719.67 seconds for the tuning loop).


In [14]:
# Get best result
best_result = results.get_best_result(metric="loss", mode="min")
print("\n" + "="*50)
print("Best hyperparameters found:")
print("="*50)
for key, value in best_result.config.items():
    print(f"{key}: {value}")
print(f"\nBest validation loss: {best_result.metrics['loss']:.6f}")
print("="*50)


Best hyperparameters found:
hidden_sizes: [64, 128, 64]
lr: 0.0017965430176632073
batch_size: 256
epochs: 150

Best validation loss: 0.004227


In [15]:
# Train final model with best config
print("\nTraining final model with best hyperparameters...")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
best_cfg = best_result.config

mlp_model = DeepMLP_Tunable(10, 14, best_cfg["hidden_sizes"]).to(device)
optimizer = torch.optim.Adam(mlp_model.parameters(), lr=best_cfg["lr"])
criterion = nn.MSELoss()

batch_size = best_cfg["batch_size"]
epochs = best_cfg["epochs"]

X_train_dev = X_train_tensor.to(device)
y_train_dev = y_train_tensor.to(device)
X_val_dev = X_val_tensor.to(device)
y_val_dev = y_val_tensor.to(device)

train_loader = DataLoader(TensorDataset(X_train_dev, y_train_dev), batch_size=batch_size, shuffle=True)



Training final model with best hyperparameters...


In [16]:
best_val_loss = float("inf")
for epoch in range(epochs):
    mlp_model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = mlp_model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    mlp_model.eval()
    with torch.no_grad():
        val_preds = mlp_model(X_val_tensor.to(device))
        val_loss = criterion(val_preds, y_val_tensor.to(device)).item()
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(mlp_model.state_dict(), "best_mlp_tuned.pth")
    print(f"Epoch {epoch+1}/{epochs} | Train Loss {total_loss/len(train_loader.dataset):.6f} | Val Loss {val_loss:.6f}")


Epoch 1/150 | Train Loss 0.108081 | Val Loss 0.041506
Epoch 2/150 | Train Loss 0.035035 | Val Loss 0.028753
Epoch 3/150 | Train Loss 0.026081 | Val Loss 0.023694
Epoch 4/150 | Train Loss 0.021494 | Val Loss 0.019677
Epoch 5/150 | Train Loss 0.018521 | Val Loss 0.017397
Epoch 6/150 | Train Loss 0.016440 | Val Loss 0.015678
Epoch 7/150 | Train Loss 0.014943 | Val Loss 0.013569
Epoch 8/150 | Train Loss 0.013888 | Val Loss 0.014335
Epoch 9/150 | Train Loss 0.013109 | Val Loss 0.012423
Epoch 10/150 | Train Loss 0.012339 | Val Loss 0.012283
Epoch 11/150 | Train Loss 0.011792 | Val Loss 0.011901
Epoch 12/150 | Train Loss 0.011213 | Val Loss 0.011924
Epoch 13/150 | Train Loss 0.010673 | Val Loss 0.011219
Epoch 14/150 | Train Loss 0.010379 | Val Loss 0.010377
Epoch 15/150 | Train Loss 0.010198 | Val Loss 0.009544
Epoch 16/150 | Train Loss 0.009625 | Val Loss 0.010727
Epoch 17/150 | Train Loss 0.009595 | Val Loss 0.009265
Epoch 18/150 | Train Loss 0.009262 | Val Loss 0.009094
Epoch 19/150 | Trai

In [ ]:
# Evaluate pretrained model

import torch
import torch.nn as nn
import numpy as np
import json
from pathlib import Path
from sklearn.metrics import r2_score
import pickle

MODEL_PATH = Path("best_mlp_tuned.pth")
NORM_PATH = Path("normalization_stats.npz")
CONFIG_PATH = Path("best_config.json")
TEST_DIR = Path("testing_trajectories")

with open(CONFIG_PATH, "r") as f:
    cfg = json.load(f)

input_dim = cfg["input_dim"]
output_dim = cfg["output_dim"]
hidden_sizes = cfg["hidden_sizes"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mlp_model = DeepMLP_Tunable(input_dim, output_dim, hidden_sizes).to(device)
mlp_model.load_state_dict(torch.load("best_mlp_tuned.pth"))
mlp_model.eval()
with torch.no_grad():
    y_pred = mlp_model(X_test_tensor.to(device)).cpu().numpy()
y_true = y_test_tensor.numpy()
y_true_denorm = y_true * y_std + y_mean
y_pred_denorm = y_pred * y_std + y_mean
mse_mlp = mean_squared_error(y_true_denorm, y_pred_denorm)
r2_mlp = r2_score(y_true_denorm, y_pred_denorm)
print(f"Tuned MLP Test MSE: {mse_mlp:.6f}, R²: {r2_mlp:.4f}")


n_joints = output_dim // 2  
print("\n===============================")
print("   R² per Joint (Position/Vel)")
print("===============================")
for j in range(n_joints):
    r2_pos = r2_score(y_true[:, j], y_pred[:, j])
    r2_vel = r2_score(y_true[:, n_joints + j], y_pred[:, n_joints + j])
    mse_pos = mean_squared_error(y_true[:, j], y_pred[:, j])
    mse_vel = mean_squared_error(y_true[:, n_joints + j], y_pred[:, n_joints + j])
    print(f"Joint {j+1:2d}: R²_pos = {r2_pos:6.4f} | R²_vel = {r2_vel:6.4f}")
    print(f"Joint {j+1:2d}: MSE_pos = {mse_pos:6.4f} | MSE_vel = {mse_vel:6.4f}")

r2_global = r2_score(y_true, y_pred)
print("\nOverall R²:", round(r2_global, 4))


In [ ]:
# Save normalization stats and best config for simulation
print("\n" + "="*60)
print("Saving model artifacts for simulation...")
print("="*60)


np.savez('normalization_stats.npz',
         X_mean=X_mean,
         X_std_safe=X_std_safe,
         y_mean=y_mean,
         y_std_safe=y_std_safe)
print("✓ Saved normalization_stats.npz")


import json
best_config_dict = {
    'hidden_sizes': best_result.config['hidden_sizes'],
    'lr': best_result.config['lr'],
    'batch_size': best_result.config['batch_size'],
    'epochs': best_result.config['epochs'],
    'best_val_loss': best_result.metrics['loss'],
    'input_dim': 10,
    'output_dim': 14
}

with open('best_config.json', 'w') as f:
    json.dump(best_config_dict, f, indent=2)
print("✓ Saved best_config.json")

print("\nModel artifacts ready for simulation:")
print("  1. best_mlp_tuned.pth (model weights)")
print("  2. normalization_stats.npz (for preprocessing)")
print("  3. best_config.json (architecture info)")
print("="*60)